<a href="https://colab.research.google.com/github/MajoRodri/HRIA/blob/feat%2Ffase1/Fase1_Exploracion_Inicial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configuración Google Colab
> **Solo si estás en Colab:** ejecuta la siguiente celda para montar Google Drive.
> Si trabajas en local (VS Code / Jupyter), puedes saltártela.

In [3]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Cambia 'archive' por el nombre de la carpeta que subiste a Drive
    RUTA_DRIVE = RUTA_DRIVE = RUTA_DRIVE = RUTA_DRIVE = '/content/drive/MyDrive/archive'

    os.chdir(RUTA_DRIVE)
    print(f"Directorio: {os.getcwd()}")
    # Instalar dependencias
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'seaborn', 'scipy'])
else:
    import os
    print(f"Entorno local — directorio: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio: /content/drive/MyDrive/archive


# EDA Proyecto I — Fase 1: Exploración Inicial
**DataTalent Solutions S.L.** | Módulo II: Análisis y Visualización de Datos

## Contexto
Somos analistas de datos junior contratados por **DataTalent Solutions S.L.**, una consultora de RRHH especializada en perfiles tech. Realizamos un EDA completo sobre el dataset **LinkedIn Job Postings** (Kaggle) para responder preguntas clave sobre el mercado laboral de roles de datos.

### Estructura del dataset
El repositorio contiene **11 archivos CSV** interrelacionados:
- **Raíz:** `postings.csv` — tabla principal (123 k+ ofertas)
- **`companies/`:** datos de empresas (4 archivos)
- **`jobs/`:** datos de las ofertas (4 archivos)
- **`mappings/`:** tablas de lookup de códigos (2 archivos)

### Preguntas de negocio
1. ¿Qué skills técnicas se piden con más frecuencia en roles de datos?
2. ¿Existen sesgos en la distribución salarial según género, ciudad o tipo de contrato?
3. ¿Qué sectores industriales concentran más ofertas y mejores salarios?
4. ¿Qué correlaciones existen entre nivel de experiencia, skills y salario?
5. ¿Qué decisiones se tomarían erróneamente con datos incompletos o sesgados?

## 1. Importación de librerías

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente")
print(f"Pandas  version: {pd.__version__}")
print(f"NumPy   version: {np.__version__}")

Librerías cargadas correctamente
Pandas  version: 2.2.2
NumPy   version: 2.0.2


## 2. Carga de los 11 archivos CSV
Cargamos todos los ficheros y presentamos un resumen de sus dimensiones.

In [5]:
# ── RAÍZ ──────────────────────────────────────────────────────────
postings             = pd.read_csv('postings.csv', low_memory=False)

# ── EMPRESAS ──────────────────────────────────────────────────────
companies            = pd.read_csv('companies/companies.csv', low_memory=False)
company_industries   = pd.read_csv('companies/company_industries.csv')
company_specialities = pd.read_csv('companies/company_specialities.csv')
employee_counts      = pd.read_csv('companies/employee_counts.csv')

# ── OFERTAS ───────────────────────────────────────────────────────
benefits             = pd.read_csv('jobs/benefits.csv')
job_industries       = pd.read_csv('jobs/job_industries.csv')
job_skills           = pd.read_csv('jobs/job_skills.csv')
salaries             = pd.read_csv('jobs/salaries.csv')

# ── MAPPINGS ──────────────────────────────────────────────────────
industries           = pd.read_csv('mappings/industries.csv')
skills_map           = pd.read_csv('mappings/skills.csv')

print(f"{'Archivo CSV':<32} {'Filas':>8} {'Columnas':>9}")
print("-" * 52)
for name, df in [
    ('postings.csv',               postings),
    ('companies.csv',              companies),
    ('company_industries.csv',     company_industries),
    ('company_specialities.csv',   company_specialities),
    ('employee_counts.csv',        employee_counts),
    ('benefits.csv',               benefits),
    ('job_industries.csv',         job_industries),
    ('job_skills.csv',             job_skills),
    ('salaries.csv',               salaries),
    ('industries.csv',             industries),
    ('skills.csv',                 skills_map),
]:
    print(f"{name:<32} {df.shape[0]:>8,} {df.shape[1]:>9}")

Archivo CSV                         Filas  Columnas
----------------------------------------------------
postings.csv                      123,849        31
companies.csv                      24,473        10
company_industries.csv             24,375         2
company_specialities.csv          169,387         2
employee_counts.csv                35,787         4
benefits.csv                       67,943         3
job_industries.csv                164,808         2
job_skills.csv                    213,768         2
salaries.csv                       40,785         8
industries.csv                        422         2
skills.csv                             35         2


## 3. Inspección del dataset principal: `postings.csv`
`postings.csv` es el **eje central**: todas las demás tablas se unen a través de `job_id` o `company_id`.

### 3.1 Dimensiones (.shape)

In [6]:
print(f"Filas    : {postings.shape[0]:,}")
print(f"Columnas : {postings.shape[1]}")
postings.head(3)

Filas    : 123,849
Columnas : 31


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0


### 3.2 Tipos de datos (.dtypes)

In [7]:
print(postings.dtypes.to_string())

job_id                          int64
company_name                   object
title                          object
description                    object
max_salary                    float64
pay_period                     object
location                       object
company_id                    float64
views                         float64
med_salary                    float64
min_salary                    float64
formatted_work_type            object
applies                       float64
original_listed_time          float64
remote_allowed                float64
job_posting_url                object
application_url                object
application_type               object
expiry                        float64
closed_time                   float64
formatted_experience_level     object
skills_desc                    object
listed_time                   float64
posting_domain                 object
sponsored                       int64
work_type                      object
currency    

### 3.3 Información general (.info())

In [8]:
postings.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

### 3.4 Estadística descriptiva — Variables numéricas (.describe())

In [9]:
postings.describe().round(2)

,job_id,max_salary,company_id,views,med_salary,min_salary,applies,original_listed_time,remote_allowed,expiry,closed_time,listed_time,sponsored,normalized_salary,zip_code,fips
count,1.238490e+05,2.979300e+04,1.221320e+05,122160.00,6280.00,29793.00,23320.00,1.238490e+05,15246.0,1.238490e+05,1.073000e+03,1.238490e+05,123849.0,3.607300e+04,102977.00,96434.00
mean,3.896402e+09,9.193942e+04,1.220401e+07,14.62,22015.62,64910.85,10.59,1.713152e+12,1.0,1.716213e+12,1.712928e+12,1.713204e+12,0.0,2.053270e+05,50400.49,28713.88
std,8.404355e+07,7.011101e+05,2.554143e+07,85.90,52255.87,495973.79,29.05,4.848209e+08,0.0,2.321394e+09,3.622893e+08,3.989122e+08,0.0,5.097627e+06,30252.23,16015.93
min,9.217160e+05,1.000000e+00,1.009000e+03,1.00,0.00,1.00,1.00,1.701811e+12,1.0,1.712903e+12,1.712346e+12,1.711317e+12,0.0,0.000000e+00,1001.00,1003.00
25%,3.894587e+09,4.828000e+01,1.435200e+04,3.00,18.94,37.00,1.00,1.712863e+12,1.0,1.715481e+12,1.712670e+12,1.712886e+12,0.0,5.200000e+04,24112.00,13121.00
50%,3.901998e+09,8.000000e+04,2.269650e+05,4.00,25.50,60000.00,3.00,1.713395e+12,1.0,1.716042e+12,1.712670e+12,1.713408e+12,0.0,8.150000e+04,48059.00,29183.00
75%,3.904707e+09,1.400000e+05,8.047188e+06,8.00,2510.50,100000.00,8.00,1.713478e+12,1.0,1.716088e+12,1.713283e+12,1.713484e+12,0.0,1.250000e+05,78201.00,42077.00
max,3.906267e+09,1.200000e+08,1.034730e+08,9975.00,750000.00,85000000.00,967.00,1.713573e+12,1.0,1.729125e+12,1.713562e+12,1.713573e+12,0.0,5.356000e+08,99901.00,56045.00


### 3.5 Estadística descriptiva — Variables categóricas

In [10]:
postings.describe(include='object').T

,count,unique,top,freq
company_name,122130,24428,Liberty Healthcare and Rehabilitation Services,1108
title,123849,72521,Sales Manager,673
description,123842,107827,Position Summary: Our Sales Manager has managi...,474
pay_period,36073,5,YEARLY,20628
location,123849,8526,United States,8125
formatted_work_type,123849,7,Full-time,98814
job_posting_url,123849,123849,https://www.linkedin.com/jobs/view/3906267224/...,1
application_url,87184,84800,https://app.dataannotation.tech/worker_signup?...,205
application_type,123849,4,OffsiteApply,84607
formatted_experience_level,94440,6,Mid-Senior level,41489


### 3.6 Identificación de variables numéricas vs. categóricas
Clasificamos cada columna para orientar las técnicas de análisis a aplicar.

In [11]:
numeric_cols     = postings.select_dtypes(include=['number']).columns.tolist()
categorical_cols = postings.select_dtypes(include=['object']).columns.tolist()

print(f"Variables NUMÉRICAS  ({len(numeric_cols)}):")
for c in numeric_cols:
    print(f"  - {c}")

print(f"\nVariables CATEGÓRICAS ({len(categorical_cols)}):")
for c in categorical_cols:
    print(f"  - {c}")

Variables NUMÉRICAS  (16):
  - job_id
  - max_salary
  - company_id
  - views
  - med_salary
  - min_salary
  - applies
  - original_listed_time
  - remote_allowed
  - expiry
  - closed_time
  - listed_time
  - sponsored
  - normalized_salary
  - zip_code
  - fips

Variables CATEGÓRICAS (15):
  - company_name
  - title
  - description
  - pay_period
  - location
  - formatted_work_type
  - job_posting_url
  - application_url
  - application_type
  - formatted_experience_level
  - skills_desc
  - posting_domain
  - work_type
  - currency
  - compensation_type


**Interpretación:** La mayoría de variables son categóricas (texto libre). Las numéricas clave son las columnas de salario (`max_salary`, `min_salary`, `med_salary`, `normalized_salary`) y métricas de interacción (`views`, `applies`). Esto implica que necesitaremos técnicas de normalización de texto además de estadística clásica.

## 4. Detección y cuantificación de valores nulos (.isnull().sum())

In [12]:
nulls   = postings.isnull().sum()
pct     = (nulls / len(postings) * 100).round(1)
null_df = pd.DataFrame({'Nulos': nulls, 'Porcentaje_%': pct})
null_df = null_df[null_df['Nulos'] > 0].sort_values('Nulos', ascending=False)

print(f"Columnas con al menos 1 nulo: {len(null_df)} de {postings.shape[1]}\n")
print(null_df.to_string())

Columnas con al menos 1 nulo: 20 de 31

                             Nulos  Porcentaje_%
closed_time                 122776          99.1
skills_desc                 121410          98.0
med_salary                  117569          94.9
remote_allowed              108603          87.7
applies                     100529          81.2
min_salary                   94056          75.9
max_salary                   94056          75.9
pay_period                   87776          70.9
compensation_type            87776          70.9
normalized_salary            87776          70.9
currency                     87776          70.9
posting_domain               39968          32.3
application_url              36665          29.6
formatted_experience_level   29409          23.7
fips                         27415          22.1
zip_code                     20872          16.9
company_name                  1719           1.4
company_id                    1717           1.4
views                        

**Interpretación crítica:**
- `med_salary`: **~95%** de nulos — prácticamente inutilizable directamente.
- `max_salary` / `min_salary`: **~76%** de nulos — analizable solo para el 24% de ofertas.
- `normalized_salary` / `pay_period` / `currency`: **~71%** de nulos.
- `remote_allowed`: **~88%** de nulos — pocas empresas indican explícitamente el teletrabajo.
- `formatted_experience_level`: **~24%** de nulos — requerirá imputación.

Este patrón probablemente **no es aleatorio (MNAR)**: las empresas con salarios menos competitivos tienden a ocultarlos estratégicamente para atraer igualmente candidatos.

## 5. Inspección de los 10 datasets secundarios

In [13]:
datasets_sec = {
    'companies':            companies,
    'company_industries':   company_industries,
    'company_specialities': company_specialities,
    'employee_counts':      employee_counts,
    'benefits':             benefits,
    'job_industries':       job_industries,
    'job_skills':           job_skills,
    'salaries':             salaries,
    'industries':           industries,
    'skills_map':           skills_map,
}

for name, df in datasets_sec.items():
    nulls_total = df.isnull().sum().sum()
    print(f"\n{'='*55}")
    print(f"  {name:<25} {df.shape[0]:>8,} filas × {df.shape[1]} cols  | {nulls_total} nulos")
    print(f"  Columnas: {list(df.columns)}")
    print(df.head(3).to_string(index=False))


  companies                   24,473 filas × 10 cols  | 3145 nulos
  Columnas: ['company_id', 'name', 'description', 'company_size', 'state', 'country', 'city', 'zip_code', 'address', 'url']
 company_id                       name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

## 6. Diagrama de relaciones entre tablas

```
postings  (job_id, company_id)
 ├── salaries            [job_id]       →  salario por oferta
 ├── job_skills          [job_id]       →  skill_abr  →  skills_map [skill_abr]
 ├── job_industries      [job_id]       →  industry_id → industries [industry_id]
 ├── benefits            [job_id]       →  beneficios por oferta
 └── companies           [company_id]   →  datos de empresa
      ├── company_industries   [company_id]
      ├── company_specialities [company_id]
      └── employee_counts      [company_id]
```

**Tipos de relación:**
| Join | Cardinalidad | Estrategia |
|------|-------------|------------|
| postings ↔ salaries | 1:1 | LEFT JOIN directo |
| postings ↔ companies | N:1 | LEFT JOIN directo |
| postings ↔ employee_counts | N:1 | LEFT JOIN (más reciente) |
| postings ↔ job_skills | 1:N | Agregar a lista separada por comas |
| postings ↔ job_industries | 1:N | Agregar a lista separada por comas |
| postings ↔ benefits | 1:N | Agregar a lista separada por comas |
| company ↔ company_industries | 1:N | Agregar a lista separada por comas |
| company ↔ company_specialities | 1:N | Agregar a lista separada por comas |

## 7. Resumen de hallazgos — Fase 1

| Aspecto | Hallazgo clave |
|---------|---------------|
| **Volumen total** | 123.849 ofertas; 11 tablas relacionadas |
| **Salario** | 71–95% de nulos — gran limitación analítica (MNAR probable) |
| **Skills** | 213.768 relaciones skill-oferta; 35 categorías distintas |
| **Industrias** | 422 industrias; 164.808 asignaciones job-industria |
| **Empresas** | 24.473 empresas con ciudad, país y sector |
| **Beneficios** | 67.943 beneficios declarados para aprox. 55% de las ofertas |
| **Experiencia** | 23.7% de nulos — requiere imputación en Fase 2 |

**Próximo paso (Fase 2):** Unir los 11 CSV en un único DataFrame maestro, limpiar, normalizar y preparar los datos para el análisis estadístico.